#### Imports and Setups 

This cell imports the Python libraries used for data handling, plotting, file paths, web parsing, Neo4j access, and helper display functions.


In [1]:
# importing libraries
import pandas as pd
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output
from IPython.display import display
import csv
import os
from pathlib import Path
import numpy as np
from bs4 import BeautifulSoup
import requests
from neo4j import GraphDatabase
import re

This cell configures pandas so long dataframe column values are shown in full when displayed.


In [2]:
# Setups
pd.set_option("display.max_colwidth", None)

This cell defines the output directory for figures related to the Tranco list analysis. It creates the directory if it does not already exist.


In [3]:
# setting figure directory 
fig_dir = Path.cwd().parent / "outputs" / "figures" / "tranco_list" 
fig_dir.mkdir(parents=True, exist_ok=True)

#### Loading Data

This cell loads the saved dependency parquet files for different TLD groups and domains with parent relationships. It keeps only the domain and unique nameserver columns needed for the IP, prefix, and AS dependency analysis.


In [ ]:
# loading dependency parquet files
base_dir = Path.cwd().parent / "data" / "result" / "dns_dependency"

try:
    gTLD_summary_df = pd.read_parquet(base_dir / "gTLD_domains_dependency_no_parent.parquet",columns=["domain","domain_unique_ns"] )
    
    ccTLD_summary_df = pd.read_parquet(base_dir / "ccTLD_domains_dependency_no_parent.parquet", columns=["domain","domain_unique_ns"] )
    
    genTLD_summary_df = pd.read_parquet(base_dir / "gen_resTLD_domains_dependency_no_parent.parquet", columns=["domain","domain_unique_ns"] )
    
    infTLD_summary_df = pd.read_parquet(base_dir / "infrTLD_domains_dependency_no_parent.parquet", columns=["domain","domain_unique_ns"] )
    
    sponTLD_summary_df = pd.read_parquet(base_dir / "sponTLD_domains_dependency_no_parent.parquet", columns=["domain","domain_unique_ns"] )

    domain_with_parent = pd.read_parquet(base_dir / "domain_with_parent_rel.parquet", columns=["domain","domain_unique_ns"] )
    
    
except Exception as e:
    print(f"failed: {e}")

##### Checking if data loaded correctly

This cell checks how many gTLD domain rows were loaded. It is a quick sanity check that the gTLD dataframe contains data.


In [5]:
len(gTLD_summary_df)

586943

This cell displays the first two rows of the ccTLD dataframe. It helps verify that the loaded columns and values look correct.


In [6]:
display(ccTLD_summary_df.head(2))

,domain,domain_unique_ns
0,nlhealthservices.ca,"[ns47.domaincontrol.com, ns48.domaincontrol.com]"
1,codesubmit.io,"[ns3.dnsimple.com, ns1.dnsimple.com, ns2.dnsimple-edge.net, ns4.dnsimple-edge.org]"


#### DNS infrastructure

I am going to use these following names and ns server to model the queries for:
1. IP
2. ASs
3. Prefix


##### Concatinating dataframes

This cell combines all TLD-group dataframes and the parent-relationship dataframe into one dataframe for analysis. It then returns the shape so the combined row and column counts can be checked.


In [7]:
# To DO: concatinate Dataframes  created ealier
concatinated_df = pd.concat(
    [
        genTLD_summary_df,
        ccTLD_summary_df,
        infTLD_summary_df,
        sponTLD_summary_df,
        gTLD_summary_df,
        domain_with_parent


    ],ignore_index=True
)

concatinated_df.shape


(945848, 2)

This cell displays the first two rows of the combined dataframe. It confirms that the concatenated dataset has the expected structure before querying dependencies.


In [8]:
display(concatinated_df.head(2))

,domain,domain_unique_ns
0,link-988betgcr.pro,"[treasure.ns.cloudflare.com, fred.ns.cloudflare.com]"
1,desabet.pro,"[apollo.ns.cloudflare.com, crystal.ns.cloudflare.com]"


##### infrastructure ungrouped

This cell takes a small sample from the combined dataframe and displays it. The sample is used for testing the dependency query before running it on the full dataset.


In [9]:
# DNS infrastructure ungrouped 

lets_play_around = concatinated_df.head(10)

display(lets_play_around)

,domain,domain_unique_ns
0,link-988betgcr.pro,"[treasure.ns.cloudflare.com, fred.ns.cloudflare.com]"
1,desabet.pro,"[apollo.ns.cloudflare.com, crystal.ns.cloudflare.com]"
2,dark2web.biz,"[jeff.ns.cloudflare.com, hera.ns.cloudflare.com]"
3,chinesesex.name,"[princess.ns.cloudflare.com, otto.ns.cloudflare.com]"
4,4006.name,"[ns2.myhostadmin.net, ns1.myhostadmin.net]"
5,nonwin.pro,"[jeff.ns.cloudflare.com, mira.ns.cloudflare.com]"
6,dutamalam.pro,"[apollo.ns.cloudflare.com, crystal.ns.cloudflare.com]"
7,iniartismu.pro,"[ns2.bhhdns.com, ns1.bhhdns.com]"
8,myliga.pro,"[ns8-cloud.nic.ru, ns4-l2.nic.ru, ns3-l2.nic.ru, ns4-cloud.nic.ru, ns8-l2.nic.ru]"
9,lensajitu.biz,"[crystal.ns.cloudflare.com, apollo.ns.cloudflare.com]"


#### Path, Db, Query & Function

This cell defines and creates the output folder where dependency query results will be saved. The try/except block reports any failure while creating the directory.


In [ ]:
# Defining folders to save the results of the queries for future use
try:
    base_dir = Path.cwd().parent / "data" / "result" 
    base_dir.mkdir(parents=True, exist_ok=True)
except Exception as e:
    print(f"failed:{e}")

This cell configures the Neo4j connection details, creates the database driver, and verifies that the connection works.


In [11]:
# Database connection details
URI = 'neo4j://localhost:7687'
AUTH = ('neo4j', 'password')
db = GraphDatabase.driver(URI, auth=AUTH)
db.verify_connectivity()


This cell defines the Cypher query used to trace each nameserver to related IPs, BGP prefixes, and autonomous systems. The query returns counts and distinct values for IP, prefix, IPv4, and AS dependencies.


In [37]:
query = """

WITH $ns_list AS ns_list

UNWIND ns_list AS ns_name
OPTIONAL MATCH (ns:AuthoritativeNameServer {name: ns_name})

CALL (ns){
    CALL apoc.path.expandConfig(ns, {
        relationshipFilter: "RESOLVES_TO_SOURCE_OPENINTEL>|PART_OF_SOURCE_IYP>|<ORIGINATE_SOURCE_BGPKIT",
        minLevel: 1,
        labelFilter: "+AuthoritativeNameServer|+BGPPrefix|+AS|+IP",
        uniqueness: "NODE_PATH",
        bfs: true
    })
    YIELD path

    RETURN collect(path) AS ns_prefix_AS_path
}

WITH collect(ns_prefix_AS_path) AS nested_paths

WITH reduce(all_paths = [], paths IN nested_paths | all_paths + paths) AS ns_prefix_AS_paths

WITH
    ns_prefix_AS_paths,
    reduce(all_nodes = [], p IN ns_prefix_AS_paths | all_nodes + nodes(p)) AS all_nodes

CALL (all_nodes){
    UNWIND all_nodes AS node
    WITH DISTINCT node
    WHERE node:IP
    RETURN collect(node.ip) AS distinct_ips
}

CALL (all_nodes) {
    WITH all_nodes
    UNWIND all_nodes AS node
    WITH DISTINCT node
    WHERE node:IP AND node.af = 4
    
    RETURN size(collect(node)) AS ipv4_count
}


CALL (all_nodes){
    UNWIND all_nodes AS node
    WITH DISTINCT node
    WHERE node:BGPPrefix 
    RETURN collect(node.prefix) AS distinct_prefixes
}

CALL (all_nodes){
    UNWIND all_nodes AS node
    WITH DISTINCT node
    WHERE node:BGPPrefix AND node.af =4
    
    RETURN size(collect(node)) AS prefixAf4_count
}

CALL (all_nodes) {
    UNWIND all_nodes AS node
    WITH DISTINCT node
    WHERE node:AS
    RETURN collect(node.asn) AS distinct_ASes
}

RETURN

    // ns_prefix_AS_paths,
    {
        IPv4_count:ipv4_count,
        IP_COUNT: size(distinct_ips),
        IP: distinct_ips,

        prefixAf4_count: prefixAf4_count,
        prefix_COUNT: size(distinct_prefixes),
        prefix: distinct_prefixes,

        AS_COUNT: size(distinct_ASes),
        AS: distinct_ASes
    } AS ip_prefix_as;
"""

This cell defines helper functions for running the Neo4j dependency query over a dataframe. The functions collect dependency results per domain and save them either as one parquet file or as chunked parquet files.


In [42]:
#Finding Dipendency
def finding_dependency(db, dataFrame, query, output_path, chunk_size=None ,overwrite=True):
    """
    Run dependency query for a list of Nameservers.

    If chunk_size is None:
        saves one parquet file to output_path

    If chunk_size is given:
        saves chunk parquet files inside output_path directory
    """

    output_path = Path(output_path)

    all_rows = []
    chunk_rows = []

    total = len(dataFrame)
    last_update = time.time()
    chunk_id = 1

    if chunk_size is not None:
        output_path.mkdir(parents=True, exist_ok=True)
    else:
        output_path.parent.mkdir(parents=True, exist_ok=True)

    no_match_count = 0

    for i, row in enumerate(dataFrame.itertuples(index=False), start=1):

        rows = fetch_paths(db, row.domain_unique_ns, row.domain, query)
        if rows and rows[0].get("matched_ns_count", 0) == 0:
            no_match_count += 1

        if chunk_size is None:
            all_rows.extend(rows)
        else:
            chunk_rows.extend(rows)

            if len(chunk_rows) >= chunk_size:
                chunk_file = output_path / f"chunk_{chunk_id:05d}.parquet"

                if chunk_file.exists() and not overwrite:
                    raise FileExistsError(f"{chunk_file} already exists")

                pd.DataFrame(chunk_rows).to_parquet(chunk_file, index=False)

                chunk_rows = []
                chunk_id += 1

        now = time.time()

        if now - last_update >= 2 or i == total:
            clear_output(wait=True)
            print(f"Processing {i}/{total}: {row.domain}")
            last_update = now

    if chunk_size is None:
        if output_path.exists() and not overwrite:
            raise FileExistsError(f"{output_path} already exists")

        df = pd.DataFrame(all_rows)
        df.to_parquet(output_path, index=False)
        return df.head(3)

    else:
        if chunk_rows:
            chunk_file = output_path / f"chunk_{chunk_id:05d}.parquet"

            if chunk_file.exists() and not overwrite:
                raise FileExistsError(f"{chunk_file} already exists")

            pd.DataFrame(chunk_rows).to_parquet(chunk_file, index=False)

        return None

# Return raw Neo4j Path objects
def fetch_paths(db, ns_list, domain_name, query):
    if ns_list is None:
        ns_list = []
    elif not isinstance(ns_list, list):
        ns_list = list(ns_list)

    try:
        records, _, _ = db.execute_query(
            query,
            ns_list=ns_list
        )

        rows = []

        for r in records:
            stats = r.get("ip_prefix_as") or {}
            row = {
                "domain": domain_name,
                "AS_count": stats.get("AS_COUNT", 0),
                "prefix_count": stats.get("prefix_COUNT", 0),
                "ip_count": stats.get("IP_COUNT", 0),
                "prefixAf4_count": stats.get("prefixAf4_count", 0),
                "IPv4_count": stats.get("IPv4_count", 0),
                "prefix": stats.get("prefix", []),
                "IP": stats.get("IP", []),
                "AS": stats.get("AS", []),
            }



            rows.append(row)

        if not rows:
            return [{
                "domain": domain_name,
                "AS_count": 0,
                "prefix_count": 0,
                "ip_count": 0,
                "prefixAf4_count": 0,
                "IPv4_count": 0,
                "prefix": [],
                "IP": [],
                "AS": [],
            }]

        return rows

    except Exception as e:
        print(f"Failed for {domain_name}: {e}")
        return []

#### Findind Dependency

##### Testing the funtion 

This cell runs the dependency function on the small sample dataframe. It writes a sample parquet result so the query output can be checked before processing all domains.


In [47]:
# Getting domain with ccTLD // --- IGNORE --- parents
finding_dependency(
    db,
    lets_play_around,
    query,
    output_path=base_dir / "IP_Prefix_AS_dependency_sample.parquet",
    chunk_size=None,
    overwrite=True
)

Processing 10/10: lensajitu.biz


,domain,AS_count,prefix_count,ip_count,prefixAf4_count,IPv4_count,prefix,IP,AS
0,link-988betgcr.pro,1,9,12,6,6,"[2a06:98c1:50::/45, 2803:f800:50::/45, 108.162.194.0/24, 162.159.32.0/20, 172.64.34.0/24, 2606:4700:50::/44, 172.64.33.0/24, 108.162.193.0/24, 173.245.59.0/24]","[2a06:98c1:50::ac40:2274, 2803:f800:50::6ca2:c274, 108.162.194.116, 162.159.38.116, 172.64.34.116, 2606:4700:50::a29f:2674, 2803:f800:50::6ca2:c171, 2a06:98c1:50::ac40:2171, 172.64.33.113, 2606:4700:58::adf5:3b71, 108.162.193.113, 173.245.59.113]",[13335]
1,desabet.pro,1,9,12,6,6,"[2803:f800:50::/45, 108.162.193.0/24, 2a06:98c1:50::/45, 173.245.59.0/24, 2606:4700:50::/44, 172.64.33.0/24, 172.64.34.0/24, 162.159.32.0/20, 108.162.194.0/24]","[2803:f800:50::6ca2:c142, 108.162.193.66, 2a06:98c1:50::ac40:2142, 173.245.59.66, 2606:4700:58::adf5:3b42, 172.64.33.66, 2a06:98c1:50::ac40:22ca, 172.64.34.202, 162.159.38.202, 2606:4700:50::a29f:26ca, 108.162.194.202, 2803:f800:50::6ca2:c2ca]",[13335]
2,dark2web.biz,1,9,12,6,6,"[172.64.33.0/24, 173.245.59.0/24, 2803:f800:50::/45, 108.162.193.0/24, 2a06:98c1:50::/45, 2606:4700:50::/44, 172.64.32.0/24, 173.245.58.0/24, 108.162.192.0/24]","[172.64.33.124, 173.245.59.124, 2803:f800:50::6ca2:c17c, 108.162.193.124, 2a06:98c1:50::ac40:217c, 2606:4700:58::adf5:3b7c, 172.64.32.162, 173.245.58.162, 2803:f800:50::6ca2:c0a2, 108.162.192.162, 2606:4700:50::adf5:3aa2, 2a06:98c1:50::ac40:20a2]",[13335]


##### Finding the dependency for all resolved Names


This cell runs the IP, prefix, and AS dependency query for the full concatenated domain dataset. Results are saved in parquet chunks of 10,000 rows to make the large output easier to store and resume.


In [48]:
# Getting IP Prefix AS dependency for all the domains in the concatinated dataframe
finding_dependency(
    db,
    concatinated_df,
    query,
    output_path=base_dir / "IP_Prefix_AS_dependency",
    chunk_size=10000,
    overwrite=True
)

Processing 945848/945848: www.mygov.bd
